# Phase 3: RAG Pipeline - Energy Report Intelligence

## Objective
Build a Retrieval-Augmented Generation (RAG) pipeline that can answer questions about UK energy demand by retrieving relevant context from NESO/Ofgem reports and generating grounded answers using an LLM.

## Research Question Addressed
3. Can a RAG pipeline over NESO/Ofgem reports provide meaningful context for demand anomalies?

## What is RAG?
Retrieval-Augmented Generation is a pattern that combines two capabilities:
1. **Retrieval** - given a question, find the most relevant passages from a document corpus using semantic search (vector embeddings)
2. **Generation** - feed those passages to an LLM as context, so it can produce an answer grounded in real documents rather than relying on its training data alone

This is directly relevant to energy companies because understanding *why* demand behaves unusually requires combining quantitative data (from Phase 1-2) with qualitative knowledge locked in unstructured reports.

## Pipeline Architecture
1. **Ingest** - load PDF reports from the `docs/` folder
2. **Chunk** - split documents into passages small enough for embedding
3. **Embed** - convert each chunk into a vector using sentence-transformers (runs locally, no API needed)
4. **Store** - persist vectors in ChromaDB (local vector database, no cloud service needed)
5. **Retrieve** - given a query, find the most semantically similar chunks
6. **Generate** - pass retrieved chunks to Claude as context and generate a grounded answer

## Why ChromaDB?
We chose ChromaDB over alternatives (Pinecone, FAISS, Weaviate) because:
- Runs entirely locally - no cloud service, no account, no data leaving your machine
- Persists to disk - once documents are embedded, no need to re-embed on each run
- Simple Python API - embed, store, and query in a few lines of code
- Already installed in our environment via `pip install chromadb`

For a portfolio project, the goal is demonstrating the RAG pattern, not configuring production infrastructure.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import config
sys.path.insert(0, str(Path('..').resolve()))

# Load API key from .env
load_dotenv(Path('../.env'))
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not found in .env file'

# Paths
DOCS_DIR = Path('../docs')
DATA_DIR = Path('../data')
EMBEDDINGS_DIR = DATA_DIR / 'embeddings'

print('Environment loaded successfully')
print(f'Docs folder: {DOCS_DIR}')
print(f'PDFs found: {len(list(DOCS_DIR.glob("*.pdf")))}')
for pdf in sorted(DOCS_DIR.glob('*.pdf')):
    print(f'  - {pdf.name} ({pdf.stat().st_size / 1024 / 1024:.1f} MB)')

Environment loaded successfully
Docs folder: ..\docs
PDFs found: 10
  - demand_flexibility_service_review_2023.pdf (1.6 MB)
  - peak_demand_historical_analysis.pdf (1.9 MB)
  - summer_outlook_2023.pdf (3.0 MB)
  - summer_outlook_2024.pdf (3.0 MB)
  - winter_outlook_2020_21.pdf (1.9 MB)
  - winter_outlook_2023_24.pdf (2.4 MB)
  - winter_outlook_2024_25.pdf (16.5 MB)
  - winter_review_2021_22.pdf (1.3 MB)
  - winter_review_2023_24.pdf (5.3 MB)
  - winter_review_2024_25.pdf (8.9 MB)


In [2]:
# =============================================================================
# FORMATTED RAG DISPLAY FUNCTIONS (v2 - fixed)
# Add this cell early in your Phase 3 notebook (after imports)
# =============================================================================

import re
from IPython.display import display, HTML, Markdown


def _md_to_html(text):
    """Convert basic markdown formatting to HTML for display."""
    # Convert ## headers to bold divs
    text = re.sub(r'^## (.+)$', r'<div style="font-weight:700; font-size:14px; margin-top:12px; margin-bottom:4px;">\1</div>', text, flags=re.MULTILINE)
    # Convert **bold** to <strong>
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
    # Convert newlines to <br>
    text = text.replace('\n', '<br>')
    return text


def display_rag_result(result, show_tokens=True):
    """Display a RAG query result with clean HTML formatting."""
    question = result['question']
    answer = _md_to_html(result['answer'])
    sources = result['sources']
    
    # Build source badges
    source_html = ' '.join([
        f'<span style="background:#e8f4f8; color:#1a5276; padding:2px 8px; '
        f'border-radius:4px; font-size:12px; margin-right:4px;">'
        f'{src} p.{page}</span>'
        for src, page in sources
    ])
    
    # Token info
    token_html = ''
    if show_tokens and 'input_tokens' in result:
        token_html = (
            f'<div style="color:#888; font-size:11px; margin-top:8px;">'
            f'Tokens: {result["input_tokens"]} in, {result["output_tokens"]} out'
            f'</div>'
        )
    
    html = f'''
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0; 
                background:#fafafa;">
        <div style="font-weight:600; color:#2c3e50; font-size:14px; margin-bottom:10px;">
            Q: {question}
        </div>
        <div style="border-left:3px solid #3498db; padding-left:12px; margin:12px 0; 
                    color:#333; font-size:13px; line-height:1.6;">
            {answer}
        </div>
        <div style="margin-top:10px;">
            <span style="font-size:11px; color:#666; margin-right:6px;">Sources:</span>
            {source_html}
        </div>
        {token_html}
    </div>
    '''
    display(HTML(html))


def display_anomaly_result(date_str, peak, anomaly_type, z_score, result):
    """Display a RAG result for a specific anomaly with context header."""
    
    # Colour coding
    if anomaly_type == 'low':
        badge_color = '#2980b9'
        badge_bg = '#ebf5fb'
        arrow = '&#9660;'
    else:
        badge_color = '#c0392b'
        badge_bg = '#fdedec'
        arrow = '&#9650;'
    
    answer = _md_to_html(result['answer'])
    question = result['question']
    sources = result['sources']
    
    source_html = ' '.join([
        f'<span style="background:#e8f4f8; color:#1a5276; padding:2px 8px; '
        f'border-radius:4px; font-size:12px; margin-right:4px;">'
        f'{src} p.{page}</span>'
        for src, page in sources
    ])
    
    html = f'''
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0; 
                background:#fafafa;">
        <div style="display:flex; align-items:center; gap:12px; margin-bottom:6px;">
            <span style="font-weight:700; font-size:16px; color:#2c3e50;">{date_str}</span>
            <span style="background:{badge_bg}; color:{badge_color}; padding:3px 10px; 
                         border-radius:4px; font-size:12px; font-weight:600;">
                {arrow} {anomaly_type.upper()} (z={z_score:.2f})
            </span>
            <span style="color:#555; font-size:13px;">Peak: {peak:,.0f} MW</span>
        </div>
        <div style="color:#666; font-size:12px; font-style:italic; margin-bottom:10px;">
            RAG query: {question}
        </div>
        <div style="border-left:3px solid #3498db; padding-left:12px; margin:12px 0; 
                    color:#333; font-size:13px; line-height:1.6; word-wrap:break-word;">
            {answer}
        </div>
        <div style="margin-top:10px;">
            <span style="font-size:11px; color:#666; margin-right:6px;">Sources:</span>
            {source_html}
        </div>
        <div style="color:#888; font-size:11px; margin-top:6px;">
            Tokens: {result['input_tokens']} in, {result['output_tokens']} out
        </div>
    </div>
    '''
    display(HTML(html))


def display_retrieval_results(query, results):
    """Display retrieval results (without LLM generation) in a clean format."""
    
    rows_html = ''
    for i, r in enumerate(results):
        # Escape HTML and ensure wrapping
        text_preview = r['text'][:300].replace('<', '&lt;').replace('>', '&gt;')
        rows_html += f'''
        <div style="border-bottom:1px solid #eee; padding:8px 0;">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <span style="font-weight:600; color:#2c3e50; font-size:12px;">
                    Result {i+1}
                </span>
                <span style="background:#fdebd0; color:#935116; padding:2px 8px; 
                             border-radius:4px; font-size:11px; white-space:nowrap;">
                    distance: {r['distance']:.4f}
                </span>
            </div>
            <div style="color:#666; font-size:11px; margin:2px 0;">
                {r['source']}, Page {r['page']}
            </div>
            <div style="color:#444; font-size:12px; line-height:1.4; margin-top:4px;
                        word-wrap:break-word; overflow-wrap:break-word;">
                {text_preview}...
            </div>
        </div>
        '''
    
    html = f'''
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0;
                background:#fafafa;">
        <div style="font-weight:600; color:#2c3e50; font-size:14px; margin-bottom:12px;">
            Query: "{query}"
        </div>
        {rows_html}
    </div>
    '''
    display(HTML(html))

## 1. Document Ingestion

We load each PDF from the `docs/` folder and extract its text content. We use PyMuPDF (`pymupdf`) for extraction because it handles complex PDF layouts (tables, multi-column text, headers/footers) more reliably than simpler alternatives.

Each page is extracted as a separate text block, tagged with the source document name and page number for traceability - when the RAG system cites a passage, we can trace it back to the exact report and page.

In [3]:
import pymupdf  # PyMuPDF

def extract_pdf_pages(pdf_path):
    """Extract text from each page of a PDF, returning a list of (text, metadata) tuples."""
    doc = pymupdf.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        text = doc[page_num].get_text()
        if text.strip():  # skip blank pages
            pages.append({
                'text': text.strip(),
                'source': pdf_path.name,
                'page': page_num + 1,
            })
    doc.close()
    return pages

# Extract all documents
all_pages = []
for pdf_path in sorted(DOCS_DIR.glob('*.pdf')):
    pages = extract_pdf_pages(pdf_path)
    all_pages.extend(pages)
    print(f'{pdf_path.name}: {len(pages)} pages extracted')

print(f'\nTotal pages extracted: {len(all_pages)}')

demand_flexibility_service_review_2023.pdf: 20 pages extracted
peak_demand_historical_analysis.pdf: 47 pages extracted
summer_outlook_2023.pdf: 30 pages extracted
summer_outlook_2024.pdf: 28 pages extracted
winter_outlook_2020_21.pdf: 22 pages extracted
winter_outlook_2023_24.pdf: 26 pages extracted
winter_outlook_2024_25.pdf: 32 pages extracted
winter_review_2021_22.pdf: 26 pages extracted
winter_review_2023_24.pdf: 24 pages extracted
winter_review_2024_25.pdf: 26 pages extracted

Total pages extracted: 281


## 2. Chunking

Embedding models have a maximum input length (typically 256-512 tokens). Full PDF pages are usually too long for a single embedding, so we split them into smaller chunks.

**Chunking strategy:**
- Target chunk size: ~500 characters (roughly 100-125 tokens)
- Overlap: 50 characters between chunks, so context isn't lost at chunk boundaries
- Each chunk retains the source document and page number metadata

This is a simple character-based chunking approach. More sophisticated methods (sentence-level, semantic chunking) exist but add complexity without proportional benefit for this corpus size.

In [4]:
from config.settings import CHUNK_SIZE, CHUNK_OVERLAP

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

# Chunk all pages
all_chunks = []
for page in all_pages:
    chunks = chunk_text(page['text'])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'source': page['source'],
            'page': page['page'],
            'chunk_id': f"{page['source']}_p{page['page']}_c{i}",
        })

print(f'Total chunks created: {len(all_chunks)}')
print(f'Average chunk length: {np.mean([len(c["text"]) for c in all_chunks]):.0f} characters')
print(f'\nChunks per document:')
chunk_counts = pd.Series([c['source'] for c in all_chunks]).value_counts()
for doc, count in chunk_counts.items():
    print(f'  {doc}: {count}')

Total chunks created: 1325
Average chunk length: 449 characters

Chunks per document:
  peak_demand_historical_analysis.pdf: 203
  summer_outlook_2023.pdf: 169
  summer_outlook_2024.pdf: 144
  winter_outlook_2023_24.pdf: 134
  winter_outlook_2024_25.pdf: 125
  winter_outlook_2020_21.pdf: 123
  winter_review_2021_22.pdf: 122
  winter_review_2023_24.pdf: 109
  winter_review_2024_25.pdf: 105
  demand_flexibility_service_review_2023.pdf: 91


## 3. Embedding & Vector Storage

We convert each text chunk into a dense vector (embedding) using `sentence-transformers`. The model `all-MiniLM-L6-v2` produces 384-dimensional vectors and runs entirely locally - no API calls, no cost.

These vectors capture the semantic meaning of each chunk, so that semantically similar passages end up close together in vector space. This is what makes retrieval work - we embed the query the same way and find the nearest chunks.

ChromaDB handles both the embedding and storage in one step. It persists the vectors to disk so we only need to embed once.

In [5]:
import chromadb
from config.settings import EMBEDDING_MODEL, COLLECTION_NAME

# Create persistent ChromaDB client
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(EMBEDDINGS_DIR))

# Delete existing collection if re-running (to avoid duplicates)
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except:
    pass

# Create collection with sentence-transformers embedding function
from chromadb.utils import embedding_functions
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# Add all chunks to the collection
# ChromaDB handles embedding automatically via the embedding function
collection.add(
    ids=[c['chunk_id'] for c in all_chunks],
    documents=[c['text'] for c in all_chunks],
    metadatas=[{'source': c['source'], 'page': c['page']} for c in all_chunks],
)

print(f'Embedded and stored {collection.count()} chunks in ChromaDB')
print(f'Embedding model: {EMBEDDING_MODEL}')
print(f'Collection: {COLLECTION_NAME}')
print(f'Storage: {EMBEDDINGS_DIR}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8794.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded and stored 1325 chunks in ChromaDB
Embedding model: all-MiniLM-L6-v2
Collection: energy_reports
Storage: ..\data\embeddings


## 4. Retrieval

The retrieval step takes a natural language query, embeds it using the same model, and finds the most similar chunks in the vector store. ChromaDB handles this with a single `.query()` call.

We retrieve the top 5 most relevant chunks for each query. This gives the LLM enough context to produce a grounded answer while keeping the input concise.

Let's test the retrieval system with a few queries related to the anomalies we found in Phase 1.

In [6]:
def retrieve(query, n_results=5):
    """Retrieve the most relevant chunks for a query."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
    )
    
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            'text': results['documents'][0][i],
            'source': results['metadatas'][0][i]['source'],
            'page': results['metadatas'][0][i]['page'],
            'distance': results['distances'][0][i],
        })
    return retrieved

# Test retrieval with a query related to Phase 1 anomalies
test_query = 'What factors affect UK electricity demand during winter cold snaps?'
results = retrieve(test_query)
display_retrieval_results(test_query, results)

In [7]:
# Test with another query - holiday demand
test_query_2 = 'How do public holidays and bank holidays affect electricity demand in the UK?'
results_2 = retrieve(test_query_2)
display_retrieval_results(test_query_2, results_2)

## 5. Generation (LLM-Augmented Answers)

The final step takes the retrieved passages and feeds them to Claude as context, asking it to answer the question based only on the provided documents. This is the "generation" in RAG.

We use `claude-sonnet-4` via the Anthropic API. While not the cheapest model available, it was the most accessible on our API plan and is more than capable of summarising document passages. Each query costs approximately $0.01.

The system prompt instructs the model to:
- Only use information from the provided context
- Cite the source document and page number
- Say "I don't have enough information" if the context doesn't contain a relevant answer

This grounding prevents hallucination - the model can't make up facts because it's constrained to the retrieved passages.

In [8]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

# Use claude-4-sonnet - widely available and cost-efficient for summarisation tasks
RAG_MODEL = 'claude-sonnet-4-20250514'

SYSTEM_PROMPT = """You are an energy analyst assistant. Answer questions about UK electricity 
demand and grid operations using ONLY the provided context from NESO/Ofgem reports.

Rules:
- Only use information from the provided context passages
- Cite the source document and page number for each claim
- If the context doesn't contain enough information to answer, say so clearly
- Be concise and factual
- Use specific numbers and data from the reports where available"""

def rag_query(question, n_results=5):
    """Full RAG pipeline: retrieve relevant passages, then generate an answer."""
    # Step 1: Retrieve
    passages = retrieve(question, n_results=n_results)
    
    # Step 2: Build context string
    context = '\n\n'.join([
        f'[Source: {p["source"]}, Page {p["page"]}]\n{p["text"]}'
        for p in passages
    ])
    
    # Step 3: Generate answer
    user_message = f"""Context from NESO/Ofgem reports:

{context}

Question: {question}

Answer based only on the context above:"""
    
    response = client.messages.create(
        model=RAG_MODEL,
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_message}],
    )
    
    answer = response.content[0].text
    
    return {
        'question': question,
        'answer': answer,
        'sources': [(p['source'], p['page']) for p in passages],
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
    }

print(f'RAG pipeline ready. Model: {RAG_MODEL}')

RAG pipeline ready. Model: claude-sonnet-4-20250514


In [9]:
# Test the full RAG pipeline
result = rag_query('What factors affect UK electricity demand during winter cold snaps?')
display_rag_result(result)

### 5.1 General Knowledge Queries (Showcase)

The RAG pipeline works best when the question aligns with the type of content in the corpus. Our NESO/Ofgem reports contain rich information about demand drivers, grid risks, and seasonal patterns. Let's demonstrate this with queries the corpus is well-equipped to answer.

In [10]:
# General knowledge queries - showcasing RAG strengths
general_queries = [
    'What are the main risks to UK electricity supply security during winter?',
    'How has embedded solar generation affected electricity demand patterns?',
    'What role do interconnectors play in UK electricity supply during peak demand?',
]

for q in general_queries:
    result = rag_query(q)
    display_rag_result(result)


## 6. Connecting to Phase 1 Anomalies

In Phase 1, we identified 16 anomalous demand days. A naive approach would ask the RAG system to explain specific dates (e.g., "why was demand low on 2022-01-01?"). However, our corpus contains seasonal outlook and review reports, not day-by-day operational logs.

**Hybrid approach:** Instead of asking about specific dates, we ask general questions *informed by* the anomaly pattern. For example:
- For a New Year's Day low-demand anomaly, we ask about holiday effects on grid operations
- For a March cold snap high-demand anomaly, we ask about cold weather demand drivers
- For an Easter low-demand anomaly, we ask about spring demand variability

The RAG retrieves relevant context for the general question, and we connect it to the specific anomaly in our commentary. This is a more realistic use of RAG - it augments human analysis rather than replacing it.

In [11]:
# Load the anomalies from Phase 1
demand = pd.read_csv(DATA_DIR / 'raw' / 'historic_demand.csv')
demand['SETTLEMENT_DATE'] = pd.to_datetime(demand['SETTLEMENT_DATE'], format='mixed', dayfirst=True)
demand['DATETIME'] = demand['SETTLEMENT_DATE'] + pd.to_timedelta((demand['SETTLEMENT_PERIOD'] - 1) * 30, unit='min')

# Recreate daily peak and anomaly detection from Phase 1
daily_peak = demand.groupby(demand['DATETIME'].dt.date)['ND'].max().reset_index()
daily_peak.columns = ['date', 'peak_demand']
daily_peak['date'] = pd.to_datetime(daily_peak['date'])
daily_peak['month'] = daily_peak['date'].dt.month

daily_peak['z_score'] = daily_peak.groupby('month')['peak_demand'].transform(
    lambda x: (x - x.mean()) / x.std()
)

anomalies = daily_peak[daily_peak['z_score'].abs() > 2.5].sort_values('z_score')

print(f'Anomalous days to investigate: {len(anomalies)}')
print(anomalies[['date', 'peak_demand', 'z_score']].to_string(index=False))

Anomalous days to investigate: 16
      date  peak_demand   z_score
2022-01-01        31054 -3.647174
2023-01-01        32628 -3.091910
2023-07-01        21680 -2.847382
2023-01-07        33447 -2.802989
2024-03-31        28138 -2.783691
2023-07-29        21910 -2.737072
2025-01-01        34021 -2.600498
2022-01-02        34078 -2.580390
2022-10-01        27323 -2.576222
2023-01-14        34193 -2.539821
2025-05-25        21988 -2.522003
2020-09-29        36505  2.640271
2020-09-30        36631  2.697717
2021-04-07        37842  2.746177
2020-03-05        45490  2.748882
2020-09-28        37185  2.950297


In [12]:
# Hybrid approach: general questions informed by specific anomalies
# Instead of asking about exact dates (which the corpus can't answer),
# we ask questions that the corpus CAN answer, then connect them to anomalies

hybrid_queries = [
    {
        'date': '2022-01-01', 'peak': 31054, 'type': 'low', 'z_score': -3.65,
        'query': 'What effect do public holidays and reduced industrial activity have on UK electricity demand?',
    },
    {
        'date': '2020-03-05', 'peak': 45490, 'type': 'high', 'z_score': 2.75,
        'query': 'How do cold weather events and low temperatures drive peak electricity demand in the UK?',
    },
    {
        'date': '2024-03-31', 'peak': 28138, 'type': 'low', 'z_score': -2.78,
        'query': 'What factors cause lower than expected electricity demand during spring periods?',
    },
]

for item in hybrid_queries:
    result = rag_query(item['query'])
    display_anomaly_result(item['date'], item['peak'], item['type'], item['z_score'], result)

## 7. Summary & Evaluation

### What We Built
A complete RAG pipeline that:
1. Ingests 10 NESO/Ofgem PDF reports (281 pages covering 2020-2025)
2. Chunks them into 1,325 searchable passages with source traceability
3. Embeds and stores vectors locally using ChromaDB and sentence-transformers (all-MiniLM-L6-v2)
4. Retrieves semantically relevant passages for any natural language query
5. Generates grounded answers using Claude (claude-sonnet-4), citing specific report pages

### Research Question 3: Can a RAG pipeline provide meaningful context for demand anomalies?

**Partially - with important caveats.**

The pipeline works correctly at every stage - ingestion, chunking, embedding, retrieval, and generation all function as designed. The retrieval component successfully identifies semantically relevant passages, and the LLM generates well-grounded answers that properly cite sources.

**Where it works well:** General knowledge queries about demand drivers produce substantive, well-cited answers. For example, asking about winter cold snap factors retrieves relevant passages from the peak demand research paper discussing weather, socioeconomic, technological, and behavioural demand drivers. The model synthesises these into a coherent answer with proper source citations.

**Where it falls short:** Date-specific anomaly queries (e.g., "why was demand low on 2022-01-01?") do not receive satisfactory answers. The model correctly identifies that the retrieved context lacks specific information about individual dates and honestly says so rather than hallucinating. This is the grounding working as intended - the system refuses to make up explanations.

**Root cause:** The corpus limitation, not a pipeline limitation. Our documents are primarily forward-looking outlooks and seasonal reviews that discuss expected scenarios and aggregate trends. They do not contain post-event analyses of specific days. To explain individual anomalies, we would need operational reports, daily demand summaries, or weather impact assessments that describe what actually happened on specific dates.

**This is itself a valuable finding:** It demonstrates that RAG system quality is fundamentally constrained by corpus relevance, not just pipeline engineering. A well-built RAG pipeline with the wrong documents will underperform a simpler system with the right documents.

### Pipeline Performance
- **Corpus:** 10 PDF reports, 281 pages, 1,325 chunks
- **Average chunk length:** 449 characters
- **Embedding model:** all-MiniLM-L6-v2 (384-dimensional vectors, runs locally)
- **Vector store:** ChromaDB (persistent, local)
- **Generation model:** claude-sonnet-4 (via Anthropic API)
- **Cost per query:** ~/usr/bin/bash.01 (approximately 800-900 input tokens, 250-330 output tokens)
- **Retrieval quality:** Good for general queries (distance scores 0.34-0.39), weaker for date-specific queries (distance scores 0.43-0.50 indicating less semantic similarity)

### Limitations
- **Corpus composition:** The biggest limitation. Our 10 documents are weighted toward forward-looking outlooks rather than retrospective analyses of specific events. Operational day-reports or post-event analyses would dramatically improve anomaly-specific answers.
- **Small corpus size:** 1,325 chunks from 10 documents. A production system would ingest hundreds of documents including operational reports, market updates, and weather summaries.
- **Simple chunking:** Character-based splitting may break mid-sentence or mid-table. Semantic chunking (splitting at paragraph/section boundaries) would improve retrieval quality.
- **No re-ranking:** We use raw embedding similarity. Adding a cross-encoder re-ranker after initial retrieval would improve precision.
- **No evaluation framework:** Ideally we would have a labelled set of question-answer pairs to measure retrieval accuracy (recall@k, precision@k). For a portfolio project, qualitative assessment of the answers is acceptable.
- **Single embedding model:** all-MiniLM-L6-v2 is lightweight and fast but not the most accurate. Larger models like all-mpnet-base-v2 or domain-specific energy models could improve retrieval.
- **Weather data proxy:** London-only weather (carried over from Phase 2).

### Potential Improvements
- Expand the document corpus with operational/daily reports that describe specific grid events
- Implement semantic chunking (split at section/paragraph boundaries)
- Add a cross-encoder re-ranker for better retrieval precision
- Build a Streamlit UI for interactive querying
- Integrate with Phase 2 - automatically trigger RAG queries when the forecasting model prediction error exceeds a threshold
- Experiment with larger embedding models for improved retrieval accuracy
- Add evaluation metrics with a labelled question-answer test set